In [1]:
'''
Ваша задача — построить скоринговую модель (IRB-подход согласно Базельским стандартам), 
которая предсказывает Probability of Default (PD) — вероятность того, что клиент 
допустит просрочку более 90 дней (NPL 90+) в течение следующего года.

Вам предстоит решить три классические банковские проблемы:

Разрозненность данных (Data Silos): Данные не лежат в одном файле. 
У вас есть анкета клиента, его кредитная история в других банках (из БКИ) и история его 
прошлых обращений в ваш банк. Их нужно правильно агрегировать и свести (JOIN).
Дисбаланс классов: Дефолтников всегда мало (около 8%). Алгоритм нужно заставить обращать на них внимание.
Объяснимость (Explainability): Модель не может быть "черным ящиком". Отказ в кредите 
должен быть аргументирован (согласно требованиям ЦБ). Мы будем использовать аппарат SHAP.
Источник данных:
Мы используем соревнование Kaggle: "Home Credit Default Risk".
Ссылка на скачивание:https://www.kaggle.com/c/home-credit-default-risk/data
''

_IncompleteInputError: incomplete input (3857640367.py, line 1)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# TODO 1.1: Загрузите три датасета с помощью pd.read_csv()
app_data = # <ВАШ_КОД>
bureau = # <ВАШ_КОД>
prev_app = # <ВАШ_КОД>

print(f"Размер таблицы заявок: {app_data.shape}")
print(f"Размер таблицы БКИ: {bureau.shape}")
print(f"Размер таблицы прошлых заявок: {prev_app.shape}")

# TODO 1.2: Оцените дисбаланс классов в app_data.
# Выведите долю (в процентах) дефолтных (TARGET=1) и надежных (TARGET=0) клиентов.
# Используйте метод value_counts(normalize=True).
class_balance = # <ВАШ_КОД>
print(f"\nДисбаланс классов:\n{class_balance * 100}")


In [ ]:
# TODO 2.1: Агрегация данных БКИ (bureau).
# Сгруппируйте данные по клиенту ('SK_ID_CURR'). 
# Рассчитайте:
# 1. Количество кредитов в других банках (count по колонке 'SK_ID_BUREAU')
# 2. Суммарный текущий долг (sum по 'AMT_CREDIT_SUM')
# 3. Максимальную просрочку (max по 'AMT_CREDIT_MAX_OVERDUE')
bureau_agg = bureau.groupby(# <ВАШ_КОД>).agg({
    # <ВАШ_КОД>
})
bureau_agg.columns =['BUREAU_LOANS_COUNT', 'BUREAU_TOTAL_DEBT', 'BUREAU_MAX_OVERDUE']
bureau_agg.reset_index(inplace=True)

# TODO 2.2: Агрегация внутренних данных банка (prev_app).
# Сгруппируйте по 'SK_ID_CURR'.
# Рассчитайте:
# 1. Количество прошлых заявок (count по 'SK_ID_PREV')
# 2. Среднюю запрошенную сумму (mean по 'AMT_APPLICATION')
prev_agg = prev_app.groupby(# <ВАШ_КОД>).agg({
    # <ВАШ_КОД>
})
prev_agg.columns =['PREV_APPS_COUNT', 'PREV_AVG_AMT_REQUESTED']
prev_agg.reset_index(inplace=True)

# TODO 2.3: Сведение витрины данных (Data Mart).
# Присоедините к app_data сначала bureau_agg, затем prev_agg.
# Используйте pd.merge() с параметром how='left', ключом выступает 'SK_ID_CURR'.
full_df = pd.merge(app_data, # <ВАШ_КОД>)
full_df = pd.merge(full_df, # <ВАШ_КОД>)

# Заполняем образовавшиеся пропуски нулями (если нет истории, значит 0 кредитов)
cols_to_fill =['BUREAU_LOANS_COUNT', 'BUREAU_TOTAL_DEBT', 'BUREAU_MAX_OVERDUE', 'PREV_APPS_COUNT', 'PREV_AVG_AMT_REQUESTED']
full_df[cols_to_fill] = full_df[cols_to_fill].fillna(0)

print(f"Размер итоговой витрины: {full_df.shape}")


In [ ]:
# TODO 3.1: Рассчитайте Debt-to-Income (DTI).
# Отношение суммы кредита (AMT_CREDIT) к доходу клиента (AMT_INCOME_TOTAL).
full_df['DTI_RATIO'] = # <ВАШ_КОД>

# TODO 3.2: Рассчитайте Annuity-to-Income (ATI).
# Отношение ежемесячного платежа (AMT_ANNUITY) к доходу (AMT_INCOME_TOTAL).
# Это главный показатель долговой нагрузки (ПДН).
full_df['ATI_RATIO'] = # <ВАШ_КОД>

# Убираем ID клиента (алгоритму он не нужен, только навредит)
full_df.drop(columns=['SK_ID_CURR'], inplace=True)


In [ ]:
from sklearn.model_selection import train_test_split

y = full_df['TARGET']
X = full_df.drop(columns=['TARGET'])

# TODO 4.1: Разделите выборку на Train и Test (80/20).
# ВАЖНО: добавьте параметр stratify=y, чтобы сохранить пропорцию дефолтов.
X_train, X_test, y_train, y_test = # <ВАШ_КОД>

# Подготовка категориальных признаков для CatBoost
# CatBoost требует, чтобы пустые текстовые ячейки были чем-то заполнены.
cat_features = np.where(X.dtypes == object)[0].tolist()

X_train.iloc[:, cat_features] = X_train.iloc[:, cat_features].fillna('Missing_Category')
X_test.iloc[:, cat_features] = X_test.iloc[:, cat_features].fillna('Missing_Category')


In [ ]:
from catboost import CatBoostClassifier

# TODO 5.1: Инициализируйте модель CatBoost.
# Решение проблемы дисбаланса: задайте auto_class_weights='Balanced'.
# Количество итераций (деревьев): 300, random_seed=42.
model = # <ВАШ_КОД>

print("Запуск обучения модели (может занять 1-2 минуты)...")
# TODO 5.2: Обучите модель (fit), обязательно передав список cat_features.
# <ВАШ_КОД>

# TODO 5.3: Получите предсказания вероятности дефолта (Probability of Default) для X_test.
# Используйте метод predict_proba(). Нам нужен только второй столбец [:, 1] (вероятность единицы).
y_pred_proba = # <ВАШ_КОД>


In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# TODO 6.1: Рассчитайте ROC-AUC
auc = # <ВАШ_КОД>

# TODO 6.2: Рассчитайте коэффициент Джини (Gini = 2 * AUC - 1). 
# Это стандартная метрика Банка России.
gini = # <ВАШ_КОД>

print(f"ROC-AUC: {auc:.4f}")
print(f"Коэффициент Джини: {gini:.4f}")

# TODO 6.3: Принятие бизнес-решения (Порог отсечения).
# Банк решает: "Если PD > 15%, мы отказываем в кредите".
# Создайте массив предсказаний классов (1 - дефолт, 0 - выплата), где 1 ставится, если y_pred_proba > 0.15.
# Используйте np.where().
threshold = 0.15
y_pred_business = # <ВАШ_КОД>

# Строим матрицу ошибок (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred_business)

# Визуализация матрицы ошибок (код готов для экономии времени)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Одобрили', 'Отказали'], 
            yticklabels=['Вернул долг', 'Дефолт'])
plt.ylabel('Фактически')
plt.xlabel('Решение банка')
plt.title(f'Матрица решений (Порог {threshold*100}%)')
plt.show()

# Дискуссия с группой:
# Посмотрите на нижний левый квадрат (Отдали деньги дефолтнику - False Negative). Это прямые убытки.
# Посмотрите на верхний правый квадрат (Отказали хорошему клиенту - False Positive). Это упущенная выгода.


In [ ]:
import shap

print("Подготовка SHAP Explainer...")
# TODO 7.1: Инициализируйте shap.TreeExplainer, передав в него обученную модель.
explainer = # <ВАШ_КОД>

# Берем 500 случайных клиентов для скорости расчетов
X_sample = X_test.sample(500, random_state=42)

# TODO 7.2: Рассчитайте SHAP-значения для выборки.
shap_values = # <ВАШ_КОД>

# 1. Глобальная важность факторов (Что драйвит риск портфеля?)
print("Глобальная важность признаков:")
shap.summary_plot(shap_values, X_sample)

# 2. Локальное объяснение (Печатаем причину отказа)
# Найдем клиента с самым высоким риском в выборке
client_idx = np.argmax(model.predict_proba(X_sample)[:, 1])
client_pd = model.predict_proba(X_sample)[client_idx, 1]

print(f"\nАнализ клиента № {client_idx}. Вероятность дефолта (PD): {client_pd:.2%}")

# Обязательно инициализируем JS для отрисовки
shap.initjs()

# TODO 7.3: Вызовите shap.force_plot() для конкретного клиента.
# Аргументы: 
# 1. Базовое значение: explainer.expected_value
# 2. Вектор SHAP-значений для клиента: shap_values[client_idx]
# 3. Данные клиента: X_sample.iloc[client_idx]
shap.force_plot(
    # <ВАШ_КОД>,
    # <ВАШ_КОД>,
    # <ВАШ_КОД>
)
